In [1]:
import pickle

''' set up dataloader '''
whole_dataset_dicts = []

dataset_path_task1 = '/home/saul_park/workspace/code/Baseline/dataset/ZuCo/task1-SR/pickle/task1-SR-dataset.pickle'
with open(dataset_path_task1, 'rb') as handle:
    whole_dataset_dicts.append(pickle.load(handle))

dataset_path_task2 = '/home/saul_park/workspace/code/Baseline/dataset/ZuCo/task2-NR/pickle/task2-NR-dataset.pickle'
with open(dataset_path_task2, 'rb') as handle:
    whole_dataset_dicts.append(pickle.load(handle))

dataset_path_taskNRv2 = '/home/saul_park/workspace/code/Baseline/dataset/ZuCo/task2-NR-2.0/pickle/task2-NR-2.0-dataset.pickle'
with open(dataset_path_taskNRv2, 'rb') as handle:
    whole_dataset_dicts.append(pickle.load(handle))

In [2]:
from transformers import BartTokenizer, BartForConditionalGeneration
from data import ZuCo_dataset
from torch.utils.data import DataLoader

subject_choice = 'ALL'
batch_size = 32
eeg_type_choice = 'GD'
bands_choice = ['_t1','_t2','_a1','_a2','_b1','_b2','_g1','_g2']
dataset_setting = 'unique_sent'
train_input = 'EEG'


tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')

# train dataset
train_set = ZuCo_dataset(whole_dataset_dicts, 'train', tokenizer, subject = subject_choice, eeg_type = eeg_type_choice, bands = bands_choice, setting = dataset_setting, test_input=train_input)
# dev dataset
dev_set = ZuCo_dataset(whole_dataset_dicts, 'dev', tokenizer, subject = subject_choice, eeg_type = eeg_type_choice, bands = bands_choice, setting = dataset_setting, test_input=train_input)
# test dataset
test_set = ZuCo_dataset(whole_dataset_dicts, 'test', tokenizer, subject = subject_choice, eeg_type = eeg_type_choice, bands = bands_choice, setting = dataset_setting)


dataset_sizes = {'train': len(train_set), 'dev': len(dev_set)}
print('[INFO]train_set size: ', len(train_set))
print('[INFO]dev_set size: ', len(dev_set))
print('[INFO]test_set size: ', len(test_set))

# train dataloader
train_dataloader = DataLoader(train_set, batch_size = batch_size, shuffle=True, num_workers=4)
# dev dataloader
val_dataloader = DataLoader(dev_set, batch_size = 1, shuffle=False, num_workers=4)
test_dataloader = DataLoader(test_set, batch_size = 1, shuffle=False, num_workers=4)
# dataloaders
dataloaders = {'train':train_dataloader, 'dev':val_dataloader, 'test':test_dataloader}

''' set up model '''
pretrained = BartForConditionalGeneration.from_pretrained('facebook/bart-large')

/home/saul_park/miniconda3/envs/EEGToText/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO]loading 3 task datasets
[INFO]using subjects:  ['ZAB', 'ZDM', 'ZDN', 'ZGW', 'ZJM', 'ZJN', 'ZJS', 'ZKB', 'ZKH', 'ZKW', 'ZMG', 'ZPH']
train divider = 320
dev divider = 360
[INFO]initializing a train set...
++ adding task to dataset, now we have: 3609
[INFO]using subjects:  ['ZAB', 'ZDM', 'ZDN', 'ZGW', 'ZJM', 'ZJN', 'ZJS', 'ZKB', 'ZKH', 'ZKW', 'ZMG', 'ZPH']
train divider = 240
dev divider = 270
[INFO]initializing a train set...
++ adding task to dataset, now we have: 6254
[INFO]using subjects:  ['YDR', 'YFR', 'YFS', 'YAC', 'YDG', 'YHS', 'YMD', 'YLS', 'YRH', 'YRK', 'YSD', 'YRP', 'YSL', 'YTL', 'YAG', 'YAK', 'YIS', 'YMS']
train divider = 279
dev divider = 313
[INFO]initializing a train set...
++ adding task to dataset, now we have: 10710
[INFO]input tensor size: torch.Size([56, 840])

[INFO]loading 3 task datasets
[INFO]using subjects:  ['ZAB', 'ZDM', 'ZDN', 'ZGW', 'ZJM', 'ZJN', 'ZJS', 'ZKB', 'ZKH', 'ZKW', 'ZMG', 'ZPH']
train divider = 320
dev divider = 360
[INFO]initializing a dev set

In [3]:
for input_embeddings, seq_len, input_masks, input_mask_invert, target_ids, target_mask, sent_level_EEG in dataloaders['test']:
    break

device = 'cpu'

input_embeddings_batch = input_embeddings.to(device).float() # B, 56, 840
# input_embeddings_batch = sent_level_EEG.unsqueeze(1).to(device).float()
input_masks_batch = input_masks.to(device) # B, 56
target_ids_batch = target_ids.to(device) # B, 56
input_mask_invert_batch = input_mask_invert.to(device) # B, 56
context = target_ids_batch.clone()

target_ids_batch[target_ids_batch == tokenizer.pad_token_id] = -100 

### Generate Test

In [4]:
from model_decoding import LDMTranslator, pretrainedmodule
from ldm.autoencoder import AutoencoderKL
from ldm.openaimodel import UNetModel
import torch



vocab_size = tokenizer.vocab_size
eeg_encoder = AutoencoderKL(
    embed_dim=56,
    # embed_dim=1024,
    monitor="val/rec_loss",
    ddconfig={
        "double_z": True,
        "z_channels": 58, 
        "resolution": 256,
        "in_channels": 56,
        "out_ch": 56,
        # "in_channels": 840,
        # "out_ch": 840,
        "ch": 128,
        "ch_mult": [1, 2, 4, 4],
        "num_res_blocks": 2,
        "attn_resolutions": [],
        "dropout": 0.0
    },
    lossconfig={
        "target": "torch.nn.Identity"
    },
    vocab_size = vocab_size,
    latent_dim=1024,
    type = 'eeg'
).to(device)

unet = UNetModel(
    image_size=512,
    dims = 1, 
    # in_channels=decoder_embedding_size, 
    # out_channels=decoder_embedding_size, 
    in_channels=56, 
    out_channels=56, 
    model_channels=320, 
    attention_resolutions=[4, 2, 1], 
    num_res_blocks=2, 
    channel_mult=[1, 2, 4, 4], 
    num_heads=8, 
    use_spatial_transformer=True, 
    transformer_depth=1, 
    context_dim=512, 
    use_checkpoint=False, 
    legacy=False
    ).to(device)

pretrained_module = pretrainedmodule(eeg_encoder, unet, pretrained, device).to(device)

model = LDMTranslator(pretrained=pretrained_module).to(device)

state_dict = torch.load('checkpoints/best/proposed_model_no_diffusion_main_reg_sequence_level_contrastive_projection_no_mask_0728_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.pt')
# new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)

making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 58, 32, 32) = 59392 dimensions.
making attention of type 'vanilla' with 512 in_channels


/home/saul_park/workspace/code/EEG-TEXT-Sejin/ldm/diffusion_module.py:39: RuntimeWarning: divide by zero encountered in true_divide
  lvlb_weights = betas ** 2 / (


<All keys matched successfully>

In [ ]:
from model_decoding import LDMTranslator, pretrainedmodule
from ldm.autoencoder import AutoencoderKL
from ldm.openaimodel import UNetModel
import torch



vocab_size = tokenizer.vocab_size
eeg_encoder = AutoencoderKL(
    embed_dim=56,
    # embed_dim=1024,
    monitor="val/rec_loss",
    ddconfig={
        "double_z": True,
        "z_channels": 58, 
        "resolution": 256,
        "in_channels": 56,
        "out_ch": 56,
        # "in_channels": 840,
        # "out_ch": 840,
        "ch": 128,
        "ch_mult": [1, 2, 4, 4],
        "num_res_blocks": 2,
        "attn_resolutions": [],
        "dropout": 0.0
    },
    lossconfig={
        "target": "torch.nn.Identity"
    },
    vocab_size = vocab_size,
    latent_dim=1024,
    type = 'eeg'
).to(device)

unet = UNetModel(
    image_size=512,
    dims = 1, 
    # in_channels=decoder_embedding_size, 
    # out_channels=decoder_embedding_size, 
    in_channels=56, 
    out_channels=56, 
    model_channels=320, 
    attention_resolutions=[4, 2, 1], 
    num_res_blocks=2, 
    channel_mult=[1, 2, 4, 4], 
    num_heads=8, 
    use_spatial_transformer=True, 
    transformer_depth=1, 
    context_dim=512, 
    use_checkpoint=False, 
    legacy=False
    ).to(device)

pretrained_module = pretrainedmodule(eeg_encoder, unet, pretrained, device).to(device)

model = LDMTranslator(pretrained=pretrained_module).to(device)


making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 58, 32, 32) = 59392 dimensions.
making attention of type 'vanilla' with 512 in_channels


<All keys matched successfully>

In [13]:
state_dict = torch.load('checkpoints/best/proposed_model_no_diffusion_main_contrastive_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.pt')
# new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)

print(model)

LDMTranslator(
  (layer): pretrainedmodule(
    (contrastive_learning): ContrastiveLatentLoss()
    (encoder): AutoencoderKL(
      (embedding_layer): Embedding(50265, 840)
      (fc1): Linear(in_features=64, out_features=56, bias=True)
      (encoder): Encoder(
        (conv_in): Conv1d(56, 128, kernel_size=(3,), stride=(1,), padding=(1,))
        (down): ModuleList(
          (0): Module(
            (block): ModuleList(
              (0): ResnetBlock(
                (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
                (conv1): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
                (norm2): GroupNorm(32, 128, eps=1e-06, affine=True)
                (dropout): Dropout(p=0.0, inplace=False)
                (conv2): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
              )
              (1): ResnetBlock(
                (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
                (conv1): Conv1d(128, 128, kernel_size=(3,)

In [ ]:
state_dict = torch.load('previous_checkpoint/checkpoints/best/pretrain_autoencoder_unet_no_diffusion_original_venv_contrastive_learning_with_mask_alpha0.01_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.pt')
# new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)

print(model)

LDMTranslator(
  (layer): pretrainedmodule(
    (contrastive_learning): ContrastiveLatentLoss()
    (encoder): AutoencoderKL(
      (embedding_layer): Embedding(50265, 840)
      (fc1): Linear(in_features=64, out_features=56, bias=True)
      (encoder): Encoder(
        (conv_in): Conv1d(56, 128, kernel_size=(3,), stride=(1,), padding=(1,))
        (down): ModuleList(
          (0): Module(
            (block): ModuleList(
              (0): ResnetBlock(
                (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
                (conv1): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
                (norm2): GroupNorm(32, 128, eps=1e-06, affine=True)
                (dropout): Dropout(p=0.0, inplace=False)
                (conv2): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
              )
              (1): ResnetBlock(
                (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
                (conv1): Conv1d(128, 128, kernel_size=(3,)

In [19]:
import re

def valid_text(s: str) -> bool:
    """알파벳·숫자 토큰이 하나 이상 남는 문장인지 확인"""
    if re.search(r'\w', s) is None:
        return ''
    return s

def remove_text_after_token(text, token='</s>'):
    # 특정 토큰 이후의 텍스트를 찾아 제거
    token_index = text.find(token)
    if token_index != -1:  # 토큰이 발견된 경우
        return text[:token_index]  # 토큰 이전까지의 텍스트 반환
    return text  # 토큰이 없으면 원본 텍스트 반환

seq2seqLMoutput = model.evaluation(input_embeddings_batch, input_masks_batch, input_mask_invert_batch, target_ids_batch, context) # (batch, time, n_class)
logits_previous = seq2seqLMoutput.logits
probs_previous = logits_previous[0].softmax(dim = 1)
values_previous, predictions_previous = probs_previous.topk(1)
predictions_previous = torch.squeeze(predictions_previous)
decoded_text = tokenizer.decode(predictions_previous, skip_special_tokens = True)
predicted_string_previous = remove_text_after_token(decoded_text.split('</s></s>')[0].replace('<s>',''))

predicted_string_previous = valid_text(predicted_string_previous)

print('[INFO] target string: ', tokenizer.decode(context[0], skip_special_tokens = True))
print('[INFO]prediction_string_decoded: ', decoded_text)
print('[INFO]predicted_string_previous: ', predicted_string_previous)

[INFO] target string:  Everything its title implies, a standard-issue crime drama spat out from the Tinseltown assembly line.
[INFO]prediction_string_decoded:   about predecessor implies is including man,issue, drama. between of the depthsseltowns line.g
[INFO]predicted_string_previous:   about predecessor implies is including man,issue, drama. between of the depthsseltowns line.g


In [ ]:
dummy_decoder_input_ids = torch.tensor([[tokenizer.pad_token_id]]).to(device) # B, 1
dummy_decoder_input_ids = torch.tensor([[tokenizer.bos_token_id]]).to(device) # B, 1

predictions=model.generate(input_embeddings_batch, input_masks_batch, input_mask_invert_batch, target_ids_batch,dummy_decoder_input_ids,
                            max_length=56,
                            num_beams=5,
                            do_sample=True,
                            repetition_penalty= 5.0,
                            no_repeat_ngram_size = 2,
                            
                            # num_beams=5,encoder_no_repeat_ngram_size =1,
                            # do_sample=True, top_k=15,temperature=0.5,num_return_sequences=5,
                            # early_stopping=True
                            )

predicted_string=tokenizer.batch_decode(predictions, skip_special_tokens=True)[0]
predicted_string = valid_text(predicted_string)

print('[INFO]predictions: ', predictions)
print('[INFO]predicted_string: ', predicted_string)


[INFO]predictions:  tensor([[0, 0, 2]], device='cuda:0')
[INFO]predicted_string:  


### ACL input test

In [4]:
import torch
sent_level_EEG = sent_level_EEG.unsqueeze(1)
input_embeddings_batch = torch.cat((input_embeddings, sent_level_EEG), dim=1).float()
print('input_embeddings_batch shape: ', input_embeddings_batch.shape)

input_embeddings_batch shape:  torch.Size([32, 57, 840])


In [5]:
from ldm.autoencoder import AutoencoderKL

vocab_size = tokenizer.vocab_size

eeg_encoder = AutoencoderKL(
    embed_dim=56,
    # embed_dim=1024,
    monitor="val/rec_loss",
    ddconfig={
        "double_z": True,
        "z_channels": 58, 
        "resolution": 256,
        "in_channels": 57,
        "out_ch": 56,
        # "in_channels": 840,
        # "out_ch": 840,
        "ch": 128,
        "ch_mult": [1, 2, 4, 4],
        "num_res_blocks": 2,
        "attn_resolutions": [],
        "dropout": 0.0
    },
    lossconfig={
        "target": "torch.nn.Identity"
    },
    vocab_size = vocab_size,
    latent_dim=1024,
    type = 'eeg'
)

making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 58, 32, 32) = 59392 dimensions.
making attention of type 'vanilla' with 512 in_channels


In [8]:
posterior = eeg_encoder.encode(input_embeddings_batch)
embedding_out = posterior.sample() # (B, 56, 105)
fc1 = torch.nn.Linear(105, 512)
embedding_out = fc1(embedding_out)  # (B, 56, 512)

print('embedding_out shape: ', embedding_out.shape)

embedding_out shape:  torch.Size([32, 56, 512])


In [10]:
target_ids[target_ids == tokenizer.pad_token_id] = -100

out = pretrained(inputs_embeds = embedding_out, attention_mask = input_masks,
                                        return_dict = True, labels = target_ids)# (B, 56, vocab_size))
print('out shape: ', out.logits.shape)

ValueError: Attention mask should be of size (32, 1, 57, 57), but is torch.Size([32, 1, 56, 56])

In [4]:
print(sent_level_EEG.shape)
print(sent_level_EEG[0])

torch.Size([32, 840])
tensor([-0.5183, -0.5753, -0.4503, -0.0238,  1.1541, -0.5956, -1.0141, -0.6133,
        -0.1595, -0.1971, -0.3961, -1.0224, -0.7589, -0.7143, -0.3128, -0.4961,
        -1.0154, -0.7052, -0.6886, -1.0054, -1.0204, -0.8656, -0.7358, -0.6574,
        -0.9086, -0.5422, -0.4501, -0.5699, -0.3383,  0.0091, -0.4900, -0.0423,
         0.1968,  0.2908,  0.4647, -0.2391, -0.0863,  1.0882,  0.5293,  0.8988,
         1.8687,  2.0916,  1.1545,  0.8406,  0.0268, -0.6050,  1.9982,  2.8672,
         2.5532,  2.0350,  1.3200,  1.8891,  2.3383,  3.1983,  3.1033,  2.6189,
         2.6491,  3.1402,  3.0805,  2.7301,  2.4876,  3.0078,  3.0233,  2.3201,
         1.2541, -0.0336, -0.6453,  2.5136,  2.8427,  2.6480,  1.6353,  0.6768,
        -0.3601,  2.4185,  2.5665,  1.9287,  0.8027,  0.1975,  2.1555,  1.9990,
         1.3564,  0.5975,  1.4236,  1.4330,  0.8492, -0.1852, -0.4165, -0.3732,
        -0.4150,  0.8879, -0.1572, -0.3924, -0.6539,  0.2514,  0.9291,  0.7133,
        -0.3622, -

### Check Unique sentence

In [51]:
for input_dataset_dict in whole_dataset_dicts:
    subjects = list(input_dataset_dict.keys())


    for key in subjects:
        set_list = set([])
        count = 0
        print(f'subject: {key}')
        for subject in input_dataset_dict[key]:
            if subject is None:
                continue
            # if subject['content'] in set_list:
            #     print('Duplicate content found for subject:', key)
            #     print(f'[index]: {set_list.index(subject["content"])}, {count}')
            #     print('Content:', subject['content'])
            set_list.add(subject['content'])
            count += 1
        print(f'subject {key} has {len(set_list)} contents.')
        print(f'duplicate count: {count - len(set_list)}')
        print('-'*50)

subject: ZAB
subject ZAB has 400 contents.
duplicate count: 0
--------------------------------------------------
subject: ZDM
subject ZDM has 400 contents.
duplicate count: 0
--------------------------------------------------
subject: ZDN
subject ZDN has 299 contents.
duplicate count: 0
--------------------------------------------------
subject: ZGW
subject ZGW has 400 contents.
duplicate count: 0
--------------------------------------------------
subject: ZJM
subject ZJM has 400 contents.
duplicate count: 0
--------------------------------------------------
subject: ZJN
subject ZJN has 400 contents.
duplicate count: 0
--------------------------------------------------
subject: ZJS
subject ZJS has 400 contents.
duplicate count: 0
--------------------------------------------------
subject: ZKB
subject ZKB has 400 contents.
duplicate count: 0
--------------------------------------------------
subject: ZKH
subject ZKH has 400 contents.
duplicate count: 0
----------------------------------

In [49]:
for input_dataset_dict in whole_dataset_dicts:
    subjects = list(input_dataset_dict.keys())
    set_list = []
    count = 0

    for key in subjects:
        print(f'subject: {key}')
        for subject in input_dataset_dict[key]:
            if subject is None:
                continue
            # if subject['content'] in set_list:
            #     print('Duplicate content found for subject:', key)
            #     print(f'[index]: {set_list.index(subject["content"])}, {count}')
            #     print('Content:', subject['content'])
            set_list.append(subject['content'])
            count += 1
        print(f'subject {key} has {len(set_list)} contents.')
        print(f'duplicate count: {count - len(set_list)}')
        print('-'*50)

subject: ZAB
subject ZAB has 400 contents.
duplicate count: 0
--------------------------------------------------
subject: ZDM
subject ZDM has 800 contents.
duplicate count: 0
--------------------------------------------------
subject: ZDN
subject ZDN has 1099 contents.
duplicate count: 0
--------------------------------------------------
subject: ZGW
subject ZGW has 1499 contents.
duplicate count: 0
--------------------------------------------------
subject: ZJM
subject ZJM has 1899 contents.
duplicate count: 0
--------------------------------------------------
subject: ZJN
subject ZJN has 2299 contents.
duplicate count: 0
--------------------------------------------------
subject: ZJS
subject ZJS has 2699 contents.
duplicate count: 0
--------------------------------------------------
subject: ZKB
subject ZKB has 3099 contents.
duplicate count: 0
--------------------------------------------------
subject: ZKH
subject ZKH has 3499 contents.
duplicate count: 0
---------------------------

### Results analysis

In [30]:
# import glob
# import os

# # 대상 폴더 경로
# # folder_path = './results_0723/'  # 원하는 폴더 경로로 바꿔주세요
# # folder_path = './results/'  # 원하는 폴더 경로로 바꿔주세요
# folder_path = '/home/saul_park/workspace/code/Baseline/results'  # 원하는 폴더 경로로 바꿔주세요


# # 해당 폴더 내의 *_results.txt로 끝나는 모든 파일 검색
# # file_pattern = os.path.join(folder_path, '*_results.txt')

# file_pattern = os.path.join(folder_path, '*.txt')
# matching_files = glob.glob(file_pattern)

import os
import glob

# 대상 폴더 경로들
folder_paths = [
    '/home/saul_park/workspace/code/Baseline/results',
    './results',
    '/home/saul_park/workspace/code/Dewave_Implementation/results',

    # 필요한 다른 경로도 추가 가능
]

# 매칭된 파일 리스트
matching_files = []

# 각 폴더에 대해 *.txt 파일 검색
for folder_path in folder_paths:
    file_pattern = os.path.join(folder_path, '*.txt')
    matching_files.extend(glob.glob(file_pattern))

# 결과 확인 (옵션)
print(f"총 {len(matching_files)}개 파일 찾음:")
for f in matching_files:
    print(f)


총 11개 파일 찾음:
/home/saul_park/workspace/code/Baseline/results/contrastive_2step_task1_task2_taskNRv2-BrainTranslator-all_decoding_results.txt
/home/saul_park/workspace/code/Baseline/results/Baseline_task1_task2_taskNRv2-BrainTranslator-all_decoding_results.txt
./results/proposed_model_no_diffusion_main_reg_sequence_level_contrastive_projection_no_mask_0728_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt
./results/proposed_model_no_diffusion_reg_ablation_without_pretrain_unet_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt
./results/proposed_model_no_diffusion_reg_ablation_without_unet_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt
./results/proposed_model_no_diffusion_main_reg_kl_1e_9_ablation_reparameterization_0801_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt
./results/proposed_model_reg_compare_EEGNet_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt
./results/proposed_model_no_diffusion_main_reg_task1_task2_task

##### Check unique sentence

In [ ]:
# 각 파일을 열어서 작업 수행
for file_path in matching_files:
    with open(file_path, 'r') as f:
        content = f.read()
    target = []
    prediction = []
    prediction_tf = []

    for t in content.splitlines():
        if t.split(':')[0] == 'target string':
            target.append(t.split(':')[1])
        elif t.split(':')[0] == 'predicted string with tf':
            prediction_tf.append(t.split(':')[1])
        elif t.split(':')[0] == 'predicted string':
            prediction.append(t.split(':')[1])
    duplicate_list = []
    duplicate_target = set([])
    count = 0
    for i in range(len(target)):
        if prediction_tf[i] in duplicate_list:
            count += 1
        duplicate_list.append(prediction_tf[i])
        if target[i] not in duplicate_target:
            duplicate_target.add(target[i])

    print(f'File: {file_path.split("/")[-1].split("_results.txt")[0]}')
    print(f'Total targets: {len(target)}')
    print(f'Total duplicates found: {count}')
    print(f'Total unique targets: {len(duplicate_target)}')
    print(f'Total duplicates found: {count/len(target)}')
    print('='*50)
    print('\n')

File: Baseline_skipstep1_EEG_YFR
Total targets: 230
Total duplicates found: 2
Total unique targets: 228
Total duplicates found: 0.008695652173913044


File: Baselinne_skipstep1_EEG
Total targets: 1407
Total duplicates found: 1300
Total unique targets: 104
Total duplicates found: 0.923951670220327


File: Baseline_skipstep1_EEG_YDR
Total targets: 329
Total duplicates found: 5
Total unique targets: 324
Total duplicates found: 0.015197568389057751


File: proposed_model_noise_to_text_contrastive_learning_with_mask
Total targets: 1407
Total duplicates found: 1224
Total unique targets: 104
Total duplicates found: 0.8699360341151386


File: proposed_model_no_diffusion_only_YDR
Total targets: 329
Total duplicates found: 3
Total unique targets: 324
Total duplicates found: 0.00911854103343465


File: proposed_model_noise_to_text_only_YDR
Total targets: 329
Total duplicates found: 5
Total unique targets: 324
Total duplicates found: 0.015197568389057751


File: proposed_model_no_diffusion_contras

In [19]:
with open('./evaluation/proposed_model_no_diffusion_contrastive_learning_with_mask_alpha0.01_results.txt', "r") as f:
    text = f.read()
text = text.split('\n')


target = []
prediction = []
prediction_tf = []

for t in text:
    if t.split(':')[0] == 'target string':
        target.append(t.split(':')[1])
    elif t.split(':')[0] == 'predicted string with tf':
        prediction_tf.append(t.split(':')[1])
    elif t.split(':')[0] == 'predicted string':
        prediction.append(t.split(':')[1])

print('target legth', len(target))
print('prediction length', len(prediction))
print('prediction_tf length', len(prediction_tf))

target legth 1407
prediction length 1407
prediction_tf length 1407


In [29]:
duplicate_list = []
count = 0
for i in range(len(target)):
    if prediction_tf[i] in duplicate_list and  target[duplicate_list.index(prediction_tf[i])] != target[i]:
        print(f'Index: {i} and {duplicate_list.index(prediction_tf[i])}')
        print(f'target: {target[i]}') 
        print(f'Duplicate found: {prediction_tf[i]}')
        print('='*50)
        count += 1
    duplicate_list.append(prediction_tf[i])
print(f'Total duplicates found: {len(duplicate_list) - len(set(duplicate_list))} / {len(duplicate_list)}')
print(f'Percentage of duplicates: {count/len(target)}')

Total duplicates found: 1266 / 1407
Percentage of duplicates: 0.0


In [34]:
duplicate_list = []
count = 0
for i in range(len(target)):
    if prediction_tf[i] in duplicate_list:
        count += 1
    duplicate_list.append(prediction_tf[i])
print(f'{len(duplicate_list)}')
print(f'{len(set(duplicate_list))}')
print('Total duplicates found:', count)
print(f'Percentage of duplicates: {count/len(target)}')

1407
141
Total duplicates found: 1266
Percentage of duplicates: 0.8997867803837953


##### Check duplicate sentence

In [ ]:
# 각 파일을 열어서 작업 수행
for file_path in matching_files:
    with open(file_path, 'r') as f:
        content = f.read()
    target = []
    prediction = []
    prediction_tf = []

    unique_target = []
    unique_list = []
    unique_list_tf = []

    for t in content.splitlines():
        if t.split(':')[0] == 'target string':
            target.append(t.split(':')[1])
            if t.split(':')[1] not in unique_target:
                unique_target.append(t.split(':')[1])
        elif t.split(':')[0] == 'predicted string with tf':
            prediction_tf.append(t.split(':')[1])
            if t.split(':')[1] not in unique_list_tf:
                unique_list_tf.append(t.split(':')[1])
        elif t.split(':')[0] == 'predicted string':
            prediction.append(t.split(':')[1])
            if t.split(':')[1] not in unique_list:
                unique_list.append(t.split(':')[1])
    print(f'File: {file_path.split("/")[-1].split("_results.txt")[0]}')
    print('-'*50)
    print(f'Total targets: {len(target)}')
    print(f'unique target: {len(unique_target)}')
    print('-'*50)
    print(f'Total predictions: {len(prediction)}')
    print(f'unique prediction: {len(unique_list)}')
    print('-'*50)
    print(f'Total predictions with tf: {len(prediction_tf)}')
    print(f'unique prediction with tf: {len(unique_list_tf)}')
    print('='*50)

    target_counts = [0] * len(unique_target)
    prediction_counts = [0] * len(unique_list)
    prediction_tf_counts = [0] * len(unique_list_tf)
    for i in range(len(target)):
        target_counts[unique_target.index(target[i])] += 1
        prediction_counts[unique_list.index(prediction[i])] += 1
        prediction_tf_counts[unique_list_tf.index(prediction_tf[i])] += 1
    # print('Target counts:', target_counts)
    # print('Prediction counts:', prediction_counts)
    # print('Prediction with tf counts:', prediction_tf_counts)
    print('Target counts:', sum(target_counts)/len(target_counts))
    print('Prediction counts:', sum(prediction_counts)/len(prediction_counts))
    print('Prediction with tf counts:', sum(prediction_tf_counts)/len(prediction_tf_counts))
    print('='*50)
    print('\n')
    print('\n')

File: proposed_model_no_diffusion_main_contrastive_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05
--------------------------------------------------
Total targets: 1407
unique target: 104
--------------------------------------------------
Total predictions: 1407
unique prediction: 817
--------------------------------------------------
Total predictions with tf: 1407
unique prediction with tf: 112
Target counts: 13.528846153846153
Prediction counts: 1.7221542227662179
Prediction with tf counts: 12.5625




File: best_proposed_model_no_diffusion_main_reg_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05
--------------------------------------------------
Total targets: 1407
unique target: 104
--------------------------------------------------
Total predictions: 1407
unique prediction: 922
--------------------------------------------------
Total predictions with tf: 1407
unique prediction with tf: 216
Target counts: 13.528846153846153
Prediction counts: 1.526030368763557

### Entropy

In [ ]:
import collections
import math
def calculate_entropy(sentences: list[str]) -> float:
    """
    문장 분포의 엔트로피를 계산합니다.
    
    :param sentences: 문장 리스트
    :return: Entropy score
    """
    if not sentences:
        return 0.0

    # 각 문장의 빈도 계산
    counts = collections.Counter(sentences)
    total_sentences = len(sentences)
    
    entropy = 0.0
    for count in counts.values():
        p = count / total_sentences
        entropy = entropy - p * math.log2(p)
        
    return entropy


In [11]:
# 각 파일을 열어서 작업 수행
for file_path in matching_files:
    with open(file_path, 'r') as f:
        content = f.read()
    target = []
    prediction = []
    prediction_tf = []

    for t in content.splitlines():
        if t.split(':')[0] == 'target string':
            target.append(t.split(':')[1])
        elif t.split(':')[0] == 'predicted string with tf':
            prediction_tf.append(t.split(':')[1])
        elif t.split(':')[0] == 'predicted string':
            prediction.append(t.split(':')[1])

    print(f'File: {file_path.split("/")[-1].split("_results.txt")[0]}')
    print('-'*50)
    print(f'Total targets: {len(target)}')
    print('-'*50)
    print(f'Total predictions: {len(prediction)}')
    print('-'*50)
    print(f'Total predictions with tf: {len(prediction_tf)}')
    print('='*50)

    entropy = calculate_entropy(target)
    print(f'Entropy of target sentences: {entropy:.4f}')
    
    entropy_prediction = calculate_entropy(prediction)
    print(f'Entropy of predicted sentences: {entropy_prediction:.4f}')
    
    entropy_prediction_tf = calculate_entropy(prediction_tf)
    print(f'Entropy of predicted sentences with tf: {entropy_prediction_tf:.4f}')
    
    print('='*50)
    print('\n')

File: Baseline_skipstep1_EEG_YFR
--------------------------------------------------
Total targets: 230
--------------------------------------------------
Total predictions: 230
--------------------------------------------------
Total predictions with tf: 230
dict_values([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
Entropy of target sentences: 7.8281
dict_va

### 문장 하나의 BLEU 스코어

In [27]:
import os
import glob
import evaluate  # Hugging Face evaluate 라이브러리
from transformers import BartTokenizer
import heapq

evaluate_bleu = evaluate.load("bleu")
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')

def hf_tokenizer(text: str):
    # special-token 제거 옵션(add_special_tokens=False)은 중요
    return tokenizer.tokenize(text)
matching_files = ['results/proposed_model_no_diffusion_main_reg_sequence_level_contrastive_projection_no_mask_0728_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt',
                  'results/proposed_model_no_diffusion_main_reg_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt']
# file_path = 'results/proposed_model_no_diffusion_main_reg_sequence_level_contrastive_projection_no_mask_0728_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt'


for file_path in matching_files:
    with open(file_path, 'r') as f:
        content = f.read()
    target = []
    prediction = []
    prediction_tf = []

    bleu_score = {}
    print(f'File: {file_path.split("/")[-1].split("_results.txt")[0]}')
    j=0
    evaluate_bleu_score_tf = 0
    evaluate_bleu_score = 0
    for t in content.splitlines():
        if t.split(':')[0] == 'target string':
            target.append(t.split(':')[1])
        elif t.split(':')[0] == 'predicted string with tf':
            prediction_tf.append(t.split(':')[1])
        elif t.split(':')[0] == 'predicted string':
            prediction.append(t.split(':')[1])
            for i in range(1, 5):
                score = evaluate_bleu.compute(
                    predictions=[prediction_tf[-1]],    # List[str]
                    references=[target[-1]],     # List[List[str]]
                    tokenizer=hf_tokenizer,    # 공백 기반 토크나이저 사용
                    max_order=i,
                    smooth=True
                )["bleu"]
                evaluate_bleu_score_tf += score

            # for i in range(1, 5):
            #     score = evaluate_bleu.compute(
            #         predictions=[prediction[-1]],      # List[str]
            #         references=[target[-1]],     # List[List[str]]
            #         tokenizer=hf_tokenizer,    # 공백 기반 토크나이저 사용
            #         max_order=i,
            #         smooth=True
            #     )["bleu"]
                # evaluate_bleu_score +=score
            # bleu_score[j] = evaluate_bleu_score/4 + evaluate_bleu_score_tf/4
            bleu_score[j] = evaluate_bleu_score_tf/4

            evaluate_bleu_score_tf = 0
            evaluate_bleu_score = 0
            j += 1

    # data.get을 키로 사용하여 값이 가장 큰 3개의 키를 찾습니다.
    best_keys = heapq.nlargest(10, bleu_score, key=bleu_score.get)

    print('\n\n')

    for key in best_keys:
        print(f'**key: {key}**')
        print(f'**target**: {target[key]}')
        print(f'**prediction**: {prediction[key]}')
        print('\n')
        print('-'*30)
        print('\n')
        print(f'**target**: {target[key]}')
        print(f'**prediction_tf**: {prediction_tf[key]}')
        print('\n')
        print('='*30)
        print('\n\n')

File: proposed_model_no_diffusion_main_reg_sequence_level_contrastive_projection_no_mask_0728_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt



**key: 820**
**target**:  He is a prominent member of the Bush family, the younger brother of President George W. Bush and the second son of former President George H. W. Bush and Barbara Bush.
**prediction**:  The film was directed by George Clooney, who also served as an executive producer on the television series The West Wing.


------------------------------


**target**:  He is a prominent member of the Bush family, the younger brother of President George W. Bush and the second son of former President George H. W. Bush and Barbara Bush.
**prediction_tf**:   was the member member of the American family, and son brother of President George W. Bush. the brother brother of President President George H. W. Bush. former Bush.





**key: 852**
**target**:  He is a prominent member of the Bush family, the younger brother of Presid

In [28]:
import os
import glob
import evaluate  # Hugging Face evaluate 라이브러리
from transformers import BartTokenizer
import heapq

evaluate_bleu = evaluate.load("bleu")
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')

def hf_tokenizer(text: str):
    # special-token 제거 옵션(add_special_tokens=False)은 중요
    return tokenizer.tokenize(text)
matching_files = ['results/proposed_model_no_diffusion_main_reg_sequence_level_contrastive_projection_no_mask_0728_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt',
                  'results/proposed_model_no_diffusion_main_reg_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt']
# file_path = 'results/proposed_model_no_diffusion_main_reg_sequence_level_contrastive_projection_no_mask_0728_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt'


for file_path in matching_files:
    with open(file_path, 'r') as f:
        content = f.read()
    target = []
    prediction = []
    prediction_tf = []

    bleu_score = {}
    print(f'File: {file_path.split("/")[-1].split("_results.txt")[0]}')
    j=0
    evaluate_bleu_score_tf = 0
    evaluate_bleu_score = 0
    for t in content.splitlines():
        if t.split(':')[0] == 'target string':
            target.append(t.split(':')[1])
        elif t.split(':')[0] == 'predicted string with tf':
            prediction_tf.append(t.split(':')[1])
        elif t.split(':')[0] == 'predicted string':
            prediction.append(t.split(':')[1])
            # for i in range(1, 5):
                # score = evaluate_bleu.compute(
                #     predictions=[prediction_tf[-1]],    # List[str]
                #     references=[target[-1]],     # List[List[str]]
                #     tokenizer=hf_tokenizer,    # 공백 기반 토크나이저 사용
                #     max_order=i,
                #     smooth=True
                # )["bleu"]
                # evaluate_bleu_score_tf += score

            for i in range(1, 5):
                score = evaluate_bleu.compute(
                    predictions=[prediction[-1]],      # List[str]
                    references=[target[-1]],     # List[List[str]]
                    tokenizer=hf_tokenizer,    # 공백 기반 토크나이저 사용
                    max_order=i,
                    smooth=True
                )["bleu"]
                evaluate_bleu_score +=score
            # bleu_score[j] = evaluate_bleu_score/4 + evaluate_bleu_score_tf/4
            bleu_score[j] = evaluate_bleu_score/4

            evaluate_bleu_score_tf = 0
            evaluate_bleu_score = 0
            j += 1

    # data.get을 키로 사용하여 값이 가장 큰 3개의 키를 찾습니다.
    best_keys = heapq.nlargest(10, bleu_score, key=bleu_score.get)

    print('\n\n')

    for key in best_keys:
        print(f'**key: {key}**')
        print(f'**target**: {target[key]}')
        print(f'**prediction**: {prediction[key]}')
        print('\n')
        print('-'*30)
        print('\n')
        print(f'**target**: {target[key]}')
        print(f'**prediction_tf**: {prediction_tf[key]}')
        print('\n')
        print('='*30)
        print('\n\n')

File: proposed_model_no_diffusion_main_reg_sequence_level_contrastive_projection_no_mask_0728_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt



**key: 1077**
**target**:  He was born in Overisel, Michigan.
**prediction**:  He was born in New York City, the youngest of five children.


------------------------------


**target**:  He was born in Overisel, Michigan.
**prediction_tf**:   was born in Newland, Germany,





**key: 1246**
**target**:  He was born in Overisel, Michigan.
**prediction**:  He was born in New York City, and grew up in the Bronx.


------------------------------


**target**:  He was born in Overisel, Michigan.
**prediction_tf**:   was born in Newland, Germany,





**key: 1314**
**target**:  He was born in Overisel, Michigan.
**prediction**:  He was born in New York City and grew up in the Bronx.


------------------------------


**target**:  He was born in Overisel, Michigan.
**prediction_tf**:   was born in Newland, Germany,





**key: 1111**
*

### Cherry Pickking

In [ ]:
from nltk.translate.bleu_score import corpus_bleu
import math

def calculate_corpus_bleu(predictions, references):
    weights_list = [(1.0,),(0.5,0.5),(1./3.,1./3.,1./3.),(0.25,0.25,0.25,0.25)]
    corpus_bleu_scores = []
    for weight in weights_list:
        # print('weight:',weight)
        corpus_bleu_score = corpus_bleu(references, predictions, weights = weight)
        corpus_bleu_scores.append(corpus_bleu_score)
    product = math.prod(corpus_bleu_scores)
    return product ** (1.0 / len(weights_list))


matching_files = ['results/proposed_model_no_diffusion_main_reg_sequence_level_contrastive_projection_no_mask_0728_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt',
                  'results/proposed_model_no_diffusion_main_reg_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt']
# 각 파일을 열어서 작업 수행
for file_path in matching_files:
    with open(file_path, 'r') as f:
        content = f.read()
    target = []
    prediction = []
    prediction_tf = []
    unique_target = set([]) 
    print(f'File: {file_path.split("/")[-1].split("_results.txt")[0]}')


    for t in content.splitlines():
        if t.split(':')[0] == 'target string':
            text= t.split(':')[1]
            target.append(text)
            unique_target.add(text)
        if t.split(':')[0] == 'predicted string with tf':
            text = t.split(':')[1]
            prediction_tf.append(text)
        elif t.split(':')[0] == 'predicted string':
            text = t.split(':')[1]
            prediction.append(text)

    for i, t in enumerate(unique_target):
        if file_path == 'results/proposed_model_no_diffusion_main_reg_sequence_level_contrastive_projection_no_mask_0728_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt':
            with open(f'./generated_txts/contrastive/{i}.txt', 'w') as f:
                for i in range(len(target)):
                    if target[i] == t:
                        f.write(f'target string: {t}\n')
                        f.write(f'predicted string with tf: {prediction_tf[i] / calculate_corpus_bleu([target[i]],[prediction_tf[i]])}\n')
                        f.write(f'predicted string: {prediction[i]}\n')
                        f.write('-'*50 + '\n')
                        f.write('\n\n')
        if file_path == 'results/proposed_model_no_diffusion_main_reg_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt':
            with open(f'./generated_txts/proposedmodel/{i}.txt', 'w') as f:
                for i in range(len(target)):
                    if target[i] == t:
                        f.write(f'target string: {t}\n')
                        f.write(f'predicted string with tf: {prediction_tf[i]}\n')
                        f.write(f'predicted string: {prediction[i]}\n')
                        f.write('-'*50 + '\n')
                        f.write('\n\n')

File: proposed_model_no_diffusion_main_reg_sequence_level_contrastive_projection_no_mask_0728_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt
File: proposed_model_no_diffusion_main_reg_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt


In [25]:
from nltk.translate.bleu_score import corpus_bleu
import math

def calculate_corpus_bleu(predictions, references):
    weights_list = [(1.0,),(0.5,0.5),(1./3.,1./3.,1./3.),(0.25,0.25,0.25,0.25)]
    corpus_bleu_scores = []
    for weight in weights_list:
        # print('weight:',weight)
        corpus_bleu_score = corpus_bleu(references, predictions, weights = weight)
        corpus_bleu_scores.append(corpus_bleu_score)
    product = math.prod(corpus_bleu_scores)
    return product ** (1.0 / len(weights_list))


matching_files = ['results/proposed_model_no_diffusion_main_reg_sequence_level_contrastive_projection_no_mask_0728_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt',
                  'results/proposed_model_no_diffusion_main_reg_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt']
# 각 파일을 열어서 작업 수행
for file_path in matching_files:
    with open(file_path, 'r') as f:
        content = f.read()
    target = []
    prediction = []
    prediction_tf = []
    unique_target = set([]) 
    print(f'File: {file_path.split("/")[-1].split("_results.txt")[0]}')


    for t in content.splitlines():
        if t.split(':')[0] == 'target string':
            text= t.split(':')[1]
            target.append(text)
            unique_target.add(text)
        if t.split(':')[0] == 'predicted string with tf':
            text = t.split(':')[1]
            prediction_tf.append(text)
        elif t.split(':')[0] == 'predicted string':
            text = t.split(':')[1]
            prediction.append(text)

    for i, t in enumerate(unique_target):
        unique_list = []
        unique_list_tf = []
        if file_path == 'results/proposed_model_no_diffusion_main_reg_sequence_level_contrastive_projection_no_mask_0728_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt':
            with open(f'./generated_txts/contrastive/{i}.txt', 'w') as f:
                f.write(f'target string: {t}\n')
                for i in range(len(target)):
                    if target[i] == t:
                        if prediction_tf[i] not in unique_list_tf:
                            f.write(f'predicted string with tf: {prediction_tf[i]} / {calculate_corpus_bleu([target[i]],[prediction_tf[i]])}\n')
                        if prediction[i] not in unique_list:
                            f.write(f'predicted string: {prediction[i]} / {calculate_corpus_bleu([target[i]],[prediction[i]])}\n')
                        f.write('\n\n')
                    unique_list_tf.append(prediction_tf[i])
                    unique_list.append(prediction[i])
        if file_path == 'results/proposed_model_no_diffusion_main_reg_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt':
            with open(f'./generated_txts/proposedmodel/{i}.txt', 'w') as f:
                f.write(f'target string: {t}\n')
                for i in range(len(target)):
                    if target[i] == t:
                        if prediction_tf[i] not in unique_list_tf:
                            f.write(f'predicted string with tf: {prediction_tf[i]} / {calculate_corpus_bleu([target[i]],[prediction_tf[i]])}\n')
                        if prediction[i] not in unique_list:
                            f.write(f'predicted string: {prediction[i]} / {calculate_corpus_bleu([target[i]],[prediction[i]])}\n')
                        f.write('\n\n')
                    unique_list_tf.append(prediction_tf[i])
                    unique_list.append(prediction[i])

File: proposed_model_no_diffusion_main_reg_sequence_level_contrastive_projection_no_mask_0728_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt
File: proposed_model_no_diffusion_main_reg_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt


In [ ]:
from transformers import BartTokenizer

tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')

def create_ngram(word_list, n):
  """
  주어진 리스트를 바이그램(bi-gram)으로 변환하는 함수입니다.

  Args:
    word_list: 단어(토큰)들이 담긴 리스트

  Returns:
    바이그램으로 구성된 리스트 (e.g., [['단어1', '단어2'], ['단어2', '단어3']])
  """
  # zip 함수를 사용하여 현재 요소와 다음 요소를 짝지어줍니다.
  # return [list(pair) for pair in zip(word_list, word_list[1:])]
  created_ngrams = [word_list[i:i+n] for i in range(len(word_list) - n + 1)]
  
  return [' '.join([vi for vi in created_ngram]) for created_ngram in created_ngrams]

text = 'hello I\'m junior and I can cooking'


# tokenized_text = tokenizer.tokenize(text)
# print(tokenized_text)
# for i in range(1, 5):
#     print(f'Bigrams with n={i}:')
#     created_ngrams = create_bigrams(tokenized_text, i)
#     print([' '.join([vi for vi in created_ngram]) for created_ngram in created_ngrams])
# # print([vi[0] + ' ' + vi[1] for vi in create_bigrams(tokenized_text)])
# print('\n\n')
from nltk.tokenize import word_tokenize
tokenized_text = word_tokenize(text)
print(tokenized_text)
for i in range(1, 5):
    print(f'Bigrams with n={i}:')
    
    print(create_ngram(tokenized_text, i))
# print([vi[0] + ' ' + vi[1] for vi in create_bigrams(tokenized_text)])


['hello', 'I', "'m", 'junior', 'and', 'I', 'can', 'cooking']
Bigrams with n=1:
['hello', 'I', "'m", 'junior', 'and', 'I', 'can', 'cooking']
Bigrams with n=2:
['hello I', "I 'm", "'m junior", 'junior and', 'and I', 'I can', 'can cooking']
Bigrams with n=3:
["hello I 'm", "I 'm junior", "'m junior and", 'junior and I', 'and I can', 'I can cooking']
Bigrams with n=4:
["hello I 'm junior", "I 'm junior and", "'m junior and I", 'junior and I can', 'and I can cooking']


In [ ]:
import os
import glob
import evaluate  # Hugging Face evaluate 라이브러리
from transformers import BartTokenizer
import heapq
# tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')

from nltk.tokenize import word_tokenize


# 각 파일을 열어서 작업 수행
for file_path in matching_files:
    with open(file_path, 'r') as f:
        content = f.read()
    target = []
    prediction = []
    prediction_tf = []
    print(f'File: {file_path.split("/")[-1].split("_results.txt")[0]}')

    predicted_string_total_num = 0
    predicted_string_tf_total_num = 0

    unique_unigram = set()
    unique_unigram_tf = set()

    unique_vigram = set()
    unique_vigram_tf = set()
    unique_3gram = set()
    unique_3gram_tf = set()
    unique_4gram = set()
    unique_4gram_tf = set()


    for t in content.splitlines():
        # if t.split(':')[0] == 'target string':
        #     target.append(t.split(':')[1])
        if t.split(':')[0] == 'predicted string with tf':
            text = t.split(':')[1]
            prediction_tf.append(text)
            unigram_tf = word_tokenize(text)
            vigram_tf = create_ngram(unigram_tf, 2)
            three_gram_tf = create_ngram(unigram_tf, 3)
            four_gram_tf = create_ngram(unigram_tf, 4)

            for uni in unigram_tf:    
                unique_unigram_tf.add(uni)
            for vi in vigram_tf:
                unique_vigram_tf.add(vi)
            for three in three_gram_tf:
                unique_3gram_tf.add(three)
            for four in four_gram_tf:
                unique_4gram_tf.add(four)

            predicted_string_tf_total_num += len(unigram_tf)
        elif t.split(':')[0] == 'predicted string':
            text = t.split(':')[1]
            prediction.append(text)
            unigram = word_tokenize(text)
            unigram = word_tokenize(text)
            vigram = create_ngram(unigram, 2)
            three_gram = create_ngram(unigram, 3)
            four_gram = create_ngram(unigram, 4)
            for uni in unigram:
                unique_unigram.add(uni)
            for vi in vigram:
                unique_vigram.add(vi)
            for three in three_gram:
                unique_3gram.add(three)
            for four in four_gram:
                unique_4gram.add(four)

            predicted_string_total_num += len(unigram)


    def line():
        print('\n')
        print('-'* 30)
        print('\n')
        
    print(f'unigram repetition rate: {len(unique_unigram)/predicted_string_total_num}')
    print(f'vigram repetition rate: {len(unique_vigram)/predicted_string_total_num}')
    print(f'trigram repetition rate: {len(unique_3gram)/predicted_string_total_num}')
    print(f'fourgram repetition rate: {len(unique_4gram)/predicted_string_total_num}')
    line()

    print(f'unigram_tf repetition rate: {len(unique_unigram_tf)/predicted_string_tf_total_num}')
    print(f'vigram_tf repetition rate: {len(unique_vigram_tf)/predicted_string_tf_total_num}')
    print(f'trigram_tf repetition rate: {len(unique_3gram_tf)/predicted_string_tf_total_num}')
    print(f'fourgram_tf repetition rate: {len(unique_4gram_tf)/predicted_string_tf_total_num}')
    line()
    print('='*30)

File: proposed_model_no_diffusion_main_contrastive_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05
unigram repetition rate: 0.04133905170359781
vigram repetition rate: 0.11714716861250099
trigram repetition rate: 0.16761972837741243
fourgram repetition rate: 0.20459058057342547


------------------------------


unigram_tf repetition rate: 0.023203784198414727
vigram_tf repetition rate: 0.055612375351572485
trigram_tf repetition rate: 0.06513679365891076
fourgram_tf repetition rate: 0.0640181539248274


------------------------------


File: best_proposed_model_no_diffusion_main_reg_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05
unigram repetition rate: 0.0428405572755418
vigram repetition rate: 0.12012383900928793
trigram repetition rate: 0.1729876160990712
fourgram repetition rate: 0.21257739938080494


------------------------------


unigram_tf repetition rate: 0.024686645034039573
vigram_tf repetition rate: 0.06123942228160591
trigram_tf repetition rate: 0.075

In [ ]:
import os
import glob
import evaluate  # Hugging Face evaluate 라이브러리
from transformers import BartTokenizer
import heapq

evaluate_bleu = evaluate.load("bleu")
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')

def hf_tokenizer(text: str):
    # special-token 제거 옵션(add_special_tokens=False)은 중요
    return tokenizer.tokenize(text)

file_path = 'results/proposed_model_no_diffusion_main_reg_sequence_level_contrastive_projection_no_mask_0728_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt'

with open(file_path, 'r') as f:
    content = f.read()
target = []
prediction = []
prediction_tf = []
print(f'File: {file_path.split("/")[-1].split("_results.txt")[0]}')

predicted_string_total_num = 0
predicted_string_tf_total_num = 0

unique_unigram = set()
unique_unigram_tf = set()

unique_vigram = set()
unique_vigram_tf = set()

for t in content.splitlines():
    # if t.split(':')[0] == 'target string':
    #     target.append(t.split(':')[1])
    if t.split(':')[0] == 'predicted string with tf':
        text = t.split(':')[1]
        prediction_tf.append(text)
        unigram_tf = tokenizer.tokenize(text)
        vigram_tf = [vi[0] + ' ' + vi[1] for vi in create_bigrams(unigram_tf)]
        for uni in unigram_tf:    
            unique_unigram_tf.add(uni)
        for vi in vigram_tf:
            unique_vigram_tf.add(vi)

        predicted_string_tf_total_num += len(unigram_tf)
    elif t.split(':')[0] == 'predicted string':
        text = t.split(':')[1]
        prediction.append(text)
        unigram = tokenizer.tokenize(text)
        vigram = [vi[0] + ' ' + vi[1] for vi in create_bigrams(unigram)]
        for uni in unigram:
            unique_unigram.add(uni)
        for vi in vigram:
            unique_vigram.add(vi)

        predicted_string_total_num += len(unigram)


def line():
    print('\n')
    print('-'* 30)
    print('\n')

print(f'predicted_string_tf_total_num: {predicted_string_tf_total_num}')
print(f'predicted_string_total_num: {predicted_string_total_num}')
line()
print(f'unique_unigram_tf_num: {len(unique_unigram_tf)}')
print(f'unique_vigram_tf_num: {len(unique_vigram_tf)}')
line()
print(f'unique_unigram_num: {len(unique_unigram)}')
print(f'unique_vigram_num: {len(unique_vigram)}')
line()
print(f'unigram repetition_tf rate: {len(unique_unigram_tf)/predicted_string_tf_total_num}')
print(f'vigram repetition_tf rate: {len(unique_vigram_tf)/predicted_string_tf_total_num}')
line()
print(f'unigram repetition rate: {len(unique_unigram)/predicted_string_total_num}')
print(f'vigram repetition rate: {len(unique_vigram)/predicted_string_total_num}')

File: proposed_model_no_diffusion_main_reg_sequence_level_contrastive_projection_no_mask_0728_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.txt
predicted_string_tf_total_num: 36086
predicted_string_total_num: 27665


------------------------------


unique_unigram_tf_num: 808
unique_vigram_tf_num: 1970


------------------------------


unique_unigram_num: 1189
unique_vigram_num: 3217


------------------------------


unigram repetition_tf rate: 0.022390954940974338
vigram repetition_tf rate: 0.054591808457573575


------------------------------


unigram repetition rate: 0.04297849268028194
vigram repetition rate: 0.1162841135008133


In [3]:
import os
import glob
import evaluate  # Hugging Face evaluate 라이브러리
from transformers import BartTokenizer
import heapq

evaluate_bleu = evaluate.load("bleu")
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')

def hf_tokenizer(text: str):
    # special-token 제거 옵션(add_special_tokens=False)은 중요
    return tokenizer.tokenize(text)

# 각 파일을 열어서 작업 수행
for file_path in matching_files:
    with open(file_path, 'r') as f:
        content = f.read()
    target = []
    prediction = []
    prediction_tf = []
    print(f'File: {file_path.split("/")[-1].split("_results.txt")[0]}')

    predicted_string_total_num = 0
    predicted_string_tf_total_num = 0

    unique_unigram = set()
    unique_unigram_tf = set()

    unique_vigram = set()
    unique_vigram_tf = set()

    for t in content.splitlines():
        # if t.split(':')[0] == 'target string':
        #     target.append(t.split(':')[1])
        if t.split(':')[0] == 'predicted string with tf':
            text = t.split(':')[1]
            prediction_tf.append(text)
            unigram_tf = tokenizer.tokenize(text)
            vigram_tf = [vi[0] + ' ' + vi[1] for vi in create_bigrams(unigram_tf)]
            for uni in unigram_tf:    
                unique_unigram_tf.add(uni)
            for vi in vigram_tf:
                unique_vigram_tf.add(vi)

            predicted_string_tf_total_num += len(unigram_tf)
        elif t.split(':')[0] == 'predicted string':
            text = t.split(':')[1]
            prediction.append(text)
            unigram = tokenizer.tokenize(text)
            vigram = [vi[0] + ' ' + vi[1] for vi in create_bigrams(unigram)]
            for uni in unigram:
                unique_unigram.add(uni)
            for vi in vigram:
                unique_vigram.add(vi)

            predicted_string_total_num += len(unigram)


    def line():
        print('\n')
        print('-'* 30)
        print('\n')
        
    print(f'unigram repetition rate: {len(unique_unigram)/predicted_string_total_num}')
    print(f'vigram repetition rate: {len(unique_vigram)/predicted_string_total_num}')
    line()

    print(f'unigram_tf repetition rate: {len(unique_unigram_tf)/predicted_string_tf_total_num}')
    print(f'vigram_tf repetition rate: {len(unique_vigram_tf)/predicted_string_tf_total_num}')
    line()
    print('='*30)

File: proposed_model_no_diffusion_main_contrastive_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05
unigram repetition rate: 0.04319535328284685
vigram repetition rate: 0.1173810749209617


------------------------------


unigram_tf repetition rate: 0.022653721682847898
vigram_tf repetition rate: 0.0548766878696574


------------------------------


File: best_proposed_model_no_diffusion_main_reg_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05
unigram repetition rate: 0.04421134147629272
vigram repetition rate: 0.11883558027385688


------------------------------


unigram_tf repetition rate: 0.023679973349619677
vigram_tf repetition rate: 0.05965798678585309


------------------------------


File: best_proposed_model_reg_compare_EEGNet_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05
unigram repetition rate: 0.043866254989080505
vigram repetition rate: 0.12598840274117026


------------------------------


unigram_tf repetition rate: 0.022128267076652757


### Repetition

In [7]:
from nltk.tokenize import word_tokenize

def create_ngram(word_list, n):
  """
  주어진 리스트를 바이그램(bi-gram)으로 변환하는 함수입니다.

  Args:
    word_list: 단어(토큰)들이 담긴 리스트

  Returns:
    바이그램으로 구성된 리스트 (e.g., [['단어1', '단어2'], ['단어2', '단어3']])
  """
  # zip 함수를 사용하여 현재 요소와 다음 요소를 짝지어줍니다.
  # return [list(pair) for pair in zip(word_list, word_list[1:])]
  created_ngrams = [word_list[i:i+n] for i in range(len(word_list) - n + 1)]
  
  return [' '.join([vi for vi in created_ngram]) for created_ngram in created_ngrams]

# 각 파일을 열어서 작업 수행
for file_path in matching_files:
    with open(file_path, 'r') as f:
        content = f.read()
    target = []
    prediction = []
    prediction_tf = []

    unique_target = []
    unique_list = []
    unique_list_tf = []
    predicted_string_total_num = 0
    predicted_string_tf_total_num = 0

    unique_unigram = set()
    unique_unigram_tf = set()

    unique_vigram = set()
    unique_vigram_tf = set()
    unique_3gram = set()
    unique_3gram_tf = set()
    unique_4gram = set()
    unique_4gram_tf = set()

    for t in content.splitlines():
        
        if t.split(':')[0] == 'target string':
            text = t.split(':')[1]
            target.append(text)
            if text not in unique_target:
                unique_target.append(text)
        elif t.split(':')[0] == 'predicted string with tf':
            text = t.split(':')[1]
            prediction_tf.append(text)
            unigram_tf = word_tokenize(text)
            vigram_tf = create_ngram(unigram_tf, 2)
            three_gram_tf = create_ngram(unigram_tf, 3)
            four_gram_tf = create_ngram(unigram_tf, 4)

            for uni in unigram_tf:    
                unique_unigram_tf.add(uni)
            for vi in vigram_tf:
                unique_vigram_tf.add(vi)
            for three in three_gram_tf:
                unique_3gram_tf.add(three)
            for four in four_gram_tf:
                unique_4gram_tf.add(four)
            
            predicted_string_tf_total_num += len(unigram_tf)


            if text not in unique_list_tf:
                unique_list_tf.append(text)
        elif t.split(':')[0] == 'predicted string':
            text = t.split(':')[1]
            prediction.append(text)
            unigram = word_tokenize(text)
            unigram = word_tokenize(text)
            vigram = create_ngram(unigram, 2)
            three_gram = create_ngram(unigram, 3)
            four_gram = create_ngram(unigram, 4)
            for uni in unigram:
                unique_unigram.add(uni)
            for vi in vigram:
                unique_vigram.add(vi)
            for three in three_gram:
                unique_3gram.add(three)
            for four in four_gram:
                unique_4gram.add(four)

            predicted_string_total_num += len(unigram)

            if text not in unique_list:
                unique_list.append(text)

    print(f'File: {file_path.split("/")[-1].split("_results.txt")[0]}')
    print('-'*50)
    print(f'Total targets: {len(target)}')
    print(f'unique target: {len(unique_target)}')
    print('-'*50)
    print(f'Total predictions: {len(prediction)}')
    print(f'unique prediction: {len(unique_list)}')
    print('-'*50)
    print(f'Total predictions with tf: {len(prediction_tf)}')
    print(f'unique prediction with tf: {len(unique_list_tf)}')
    print('='*50)

    target_counts = [0] * len(unique_target)
    prediction_counts = [0] * len(unique_list)
    prediction_tf_counts = [0] * len(unique_list_tf)
    for i in range(len(target)):
        target_counts[unique_target.index(target[i])] += 1
        prediction_counts[unique_list.index(prediction[i])] += 1
        prediction_tf_counts[unique_list_tf.index(prediction_tf[i])] += 1
    def line():
        print('\n')
        print('-'* 30)
        print('\n')
    print('Target counts:', round(sum(target_counts)/len(target_counts), 2))
    print('Prediction counts:', round(sum(prediction_counts)/len(prediction_counts), 2))
    print('Prediction with tf counts:', round(sum(prediction_tf_counts)/len(prediction_tf_counts), 2))
    line()
        
    print(f'unigram repetition rate: {round(len(unique_unigram)/predicted_string_total_num * 100, 2)}')
    print(f'vigram repetition rate: {round(len(unique_vigram)/predicted_string_total_num * 100, 2)}')
    print(f'trigram repetition rate: {round(len(unique_3gram)/predicted_string_total_num * 100, 2)}')
    print(f'fourgram repetition rate: {round(len(unique_4gram)/predicted_string_total_num * 100, 2)}')
    line()

    print(f'unigram_tf repetition rate: {round(len(unique_unigram_tf)/predicted_string_tf_total_num * 100, 2)}')
    print(f'vigram_tf repetition rate: {round(len(unique_vigram_tf)/predicted_string_tf_total_num * 100, 2)}')
    print(f'trigram_tf repetition rate: {round(len(unique_3gram_tf)/predicted_string_tf_total_num * 100, 2)}')
    print(f'fourgram_tf repetition rate: {round(len(unique_4gram_tf)/predicted_string_tf_total_num * 100, 2)}')

    print('='*50)
    print('\n')
    print('\n')

File: contrastive_2step_task1_task2_taskNRv2-BrainTranslator-all_decoding
--------------------------------------------------
Total targets: 1407
unique target: 104
--------------------------------------------------
Total predictions: 1407
unique prediction: 824
--------------------------------------------------
Total predictions with tf: 1407
unique prediction with tf: 106
Target counts: 13.53
Prediction counts: 1.71
Prediction with tf counts: 13.27


------------------------------


unigram repetition rate: 3.93
vigram repetition rate: 11.19
trigram repetition rate: 16.18
fourgram repetition rate: 19.86


------------------------------


unigram_tf repetition rate: 2.24
vigram_tf repetition rate: 5.44
trigram_tf repetition rate: 6.45
fourgram_tf repetition rate: 6.35




File: Baseline_task1_task2_taskNRv2-BrainTranslator-all_decoding
--------------------------------------------------
Total targets: 1407
unique target: 104
--------------------------------------------------
Total predi